# Analyze DARPA 2000 Attack Phases

In [1]:
import pandas as pd

## Load Attack Data

In [2]:
dataset = "darpa2000"
scenario = "s1_inside"
data_dir = f"../data/interim/{dataset}/{scenario}/flows_unlabeled"

In [3]:
dfs = []

attack_phases = list(range(1, 6)) 
for phase in attack_phases:
    print(f"Phase: {phase}")

    df = pd.read_csv(
        f"{data_dir}/phase{phase}_flows.csv"
    )

    dfs.append(df)

Phase: 1
Phase: 2
Phase: 3
Phase: 4
Phase: 5


## Feature Engineering

In [4]:
categorical_cols = [
    "sport",
    "dport",
    "proto",
    "service",
    "conn_state", 
    "local_orig",
    "local_resp",
    "history",    
    "tunnel_parents",
    "ip_proto",   
]

In [5]:
for col in categorical_cols:
    df_col = df[col]
    df_col_count = df_col.value_counts()
    print(f"Unique values in {col}: {len(df_col_count)}")
    # print(df_col_count)

Unique values in sport: 34246
Unique values in dport: 29272
Unique values in proto: 1
Unique values in service: 2
Unique values in conn_state: 3
Unique values in local_orig: 2
Unique values in local_resp: 2
Unique values in history: 6
Unique values in tunnel_parents: 1
Unique values in ip_proto: 1


In [30]:
def analyze_phase(df, print_details=True):

    src_ips = df["src_ip"].value_counts()
    dst_ips = df["dst_ip"].value_counts()
    src_ports = df["sport"].value_counts()
    dst_ports = df["dport"].value_counts()

    if print_details:
        print("Total Flows:", len(df))

        print("\n --- IP distribution ---")
        print(f"\nSource IPs ({len(src_ips)}):")
        print(src_ips)
        print(f"\nDestination IPs ({len(dst_ips)}):")
        print(dst_ips)

        print("\n --- Port distribution ---")
        print(f"Source Ports ({len(src_ports)}):")
        print(src_ports)
        print(f"\nDestination Ports ({len(dst_ports)}):")
        print(dst_ports)

    return src_ips, dst_ips, src_ports, dst_ports

    

## Phase 1: IP Sweep

In [31]:
df_1 = dfs[0]
df_1

,flow_id,start_time,end_time,duration,src_ip,sport,dst_ip,dport,proto,service,...,local_orig,local_resp,missed_bytes,history,orig_pkts,orig_ip_bytes,resp_pkts,resp_ip_bytes,tunnel_parents,ip_proto
0,f0,9.559119e+08,9.559119e+08,0.001254,202.77.162.213,63517,172.16.115.20,53,udp,dns,...,F,T,0,Dd,1,72,1,152,-,17
1,f1,9.559119e+08,9.559119e+08,0.000822,202.77.162.213,63518,172.16.115.20,53,udp,dns,...,F,T,0,Dd,1,63,1,131,-,17


In [32]:
src_ips_1, dst_ips_1, src_ports_1, dst_ports_1 = analyze_phase(df_1)

Total Flows: 2

 --- IP distribution ---

Source IPs (1):
src_ip
202.77.162.213    2
Name: count, dtype: int64

Destination IPs (1):
dst_ip
172.16.115.20    2
Name: count, dtype: int64

 --- Port distribution ---
Source Ports (2):
sport
63517    1
63518    1
Name: count, dtype: int64

Destination Ports (1):
dport
53    2
Name: count, dtype: int64


## Phase 2: Sadmind Ping

In [33]:
df_2 = dfs[1]
df_2

,flow_id,start_time,end_time,duration,src_ip,sport,dst_ip,dport,proto,service,...,local_orig,local_resp,missed_bytes,history,orig_pkts,orig_ip_bytes,resp_pkts,resp_ip_bytes,tunnel_parents,ip_proto
0,f0,9.559127e+08,9.559127e+08,0.000000,202.77.162.213,33912,172.16.115.20,32773,udp,-,...,F,T,0,D,1,1440,0,0,-,17
1,f1,9.559127e+08,9.559127e+08,0.001311,202.77.162.213,33910,172.16.115.20,111,udp,-,...,F,T,0,Dd,1,84,1,56,-,17
2,f2,9.559127e+08,9.559127e+08,0.000000,202.77.162.213,33896,172.16.115.20,32773,udp,-,...,F,T,0,D,1,1440,0,0,-,17
3,f3,9.559127e+08,9.559127e+08,0.001515,202.77.162.213,33894,172.16.115.20,111,udp,-,...,F,T,0,Dd,1,84,1,56,-,17


In [34]:
src_ips_2, dst_ips_2, src_ports_2, dst_ports_2 = analyze_phase(df_2)

Total Flows: 4

 --- IP distribution ---

Source IPs (1):
src_ip
202.77.162.213    4
Name: count, dtype: int64

Destination IPs (1):
dst_ip
172.16.115.20    4
Name: count, dtype: int64

 --- Port distribution ---
Source Ports (4):
sport
33912    1
33910    1
33896    1
33894    1
Name: count, dtype: int64

Destination Ports (2):
dport
32773    2
111      2
Name: count, dtype: int64


In [35]:
target_ip = "172.16.112.10"
target_flows = df_2[df_2["dst_ip"] == target_ip]
target_flows

,flow_id,start_time,end_time,duration,src_ip,sport,dst_ip,dport,proto,service,...,local_orig,local_resp,missed_bytes,history,orig_pkts,orig_ip_bytes,resp_pkts,resp_ip_bytes,tunnel_parents,ip_proto


In [36]:
different_dst_ips = dst_ips_1.index.difference(dst_ips_2.index)
different_dst_ips

Index([], dtype='object', name='dst_ip')

In [37]:
# Check which homenet IPs are involved in each phase
common_dst_ips = dst_ips_1.index.intersection(dst_ips_2.index)
common_dst_ips

Index(['172.16.115.20'], dtype='object', name='dst_ip')

In [38]:
active_ips = dst_ips_1.index.intersection(src_ips_2.index)
active_ips

Index([], dtype='object')

## Phase 3: Exploiting

In [39]:
df_3 = dfs[2]
df_3

,flow_id,start_time,end_time,duration,src_ip,sport,dst_ip,dport,proto,service,...,local_orig,local_resp,missed_bytes,history,orig_pkts,orig_ip_bytes,resp_pkts,resp_ip_bytes,tunnel_parents,ip_proto
0,f0,9.559134e+08,9.559134e+08,0.081167,172.16.115.20,20,202.77.162.213,65268,tcp,ftp-data,...,T,F,0,ShAdfFa,19,764,32,43358,-,6
1,f1,9.559134e+08,9.559134e+08,0.246931,202.77.162.213,65267,172.16.115.20,21,tcp,ftp,...,F,T,0,ShAdDaFf,10,625,11,836,-,6


In [40]:
target_ip = "172.16.112.10"
target_flows = df_3[df_3["dst_ip"] == target_ip]
target_flows

,flow_id,start_time,end_time,duration,src_ip,sport,dst_ip,dport,proto,service,...,local_orig,local_resp,missed_bytes,history,orig_pkts,orig_ip_bytes,resp_pkts,resp_ip_bytes,tunnel_parents,ip_proto


In [41]:
src_ips_3, dst_ips_3, src_ports_3, dst_ports_3 = analyze_phase(df_3)

Total Flows: 2

 --- IP distribution ---

Source IPs (2):
src_ip
172.16.115.20     1
202.77.162.213    1
Name: count, dtype: int64

Destination IPs (2):
dst_ip
202.77.162.213    1
172.16.115.20     1
Name: count, dtype: int64

 --- Port distribution ---
Source Ports (2):
sport
20       1
65267    1
Name: count, dtype: int64

Destination Ports (2):
dport
65268    1
21       1
Name: count, dtype: int64


## Phase 4: Installation

In [42]:
df_4 = dfs[3]
df_4

,flow_id,start_time,end_time,duration,src_ip,sport,dst_ip,dport,proto,service,...,local_orig,local_resp,missed_bytes,history,orig_pkts,orig_ip_bytes,resp_pkts,resp_ip_bytes,tunnel_parents,ip_proto
0,f0,9.559141e+08,9.559142e+08,47.693301,202.77.162.213,32650,172.16.115.20,23,tcp,-,...,F,T,0,ShAdDaFR,40,2273,37,2623,-,6
1,f1,9.559142e+08,9.559142e+08,0.001996,172.16.115.20,33368,172.16.112.50,111,udp,-,...,T,T,0,Dd,1,84,1,56,-,17
2,f2,9.559142e+08,9.559142e+08,0.000000,172.16.115.20,33368,172.16.112.50,32773,udp,-,...,T,T,0,D,1,1440,0,0,-,17
3,f3,9.559142e+08,9.559142e+08,0.001366,172.16.115.20,33369,172.16.112.50,111,udp,-,...,T,T,0,Dd,1,84,1,56,-,17
4,f4,9.559142e+08,9.559142e+08,0.000000,172.16.115.20,33369,172.16.112.50,32773,udp,-,...,T,T,0,D,1,1440,0,0,-,17
5,f5,9.559143e+08,9.559143e+08,0.000000,172.16.112.50,33345,172.16.115.20,9325,udp,-,...,T,T,0,D,1,37,0,0,-,17
6,f6,9.559143e+08,9.559143e+08,3.534542,172.16.115.20,32841,172.16.112.50,23,tcp,-,...,T,T,0,ShAdDFaf,26,1143,23,2388,-,6
7,f7,9.559143e+08,9.559143e+08,0.166269,172.16.112.50,20,172.16.115.20,32840,tcp,ftp-data,...,T,T,0,ShAdfFa,11,444,37,45721,-,6
8,f8,9.559143e+08,9.559143e+08,0.351929,172.16.115.20,32839,172.16.112.50,21,tcp,ftp,...,T,T,0,ShAdDFaf,12,562,10,663,-,6


In [43]:
target_ip = "172.16.112.50"
target_flows = df_4[df_4["dst_ip"] == target_ip]
target_flows

,flow_id,start_time,end_time,duration,src_ip,sport,dst_ip,dport,proto,service,...,local_orig,local_resp,missed_bytes,history,orig_pkts,orig_ip_bytes,resp_pkts,resp_ip_bytes,tunnel_parents,ip_proto
1,f1,9.559142e+08,9.559142e+08,0.001996,172.16.115.20,33368,172.16.112.50,111,udp,-,...,T,T,0,Dd,1,84,1,56,-,17
2,f2,9.559142e+08,9.559142e+08,0.000000,172.16.115.20,33368,172.16.112.50,32773,udp,-,...,T,T,0,D,1,1440,0,0,-,17
3,f3,9.559142e+08,9.559142e+08,0.001366,172.16.115.20,33369,172.16.112.50,111,udp,-,...,T,T,0,Dd,1,84,1,56,-,17
4,f4,9.559142e+08,9.559142e+08,0.000000,172.16.115.20,33369,172.16.112.50,32773,udp,-,...,T,T,0,D,1,1440,0,0,-,17
6,f6,9.559143e+08,9.559143e+08,3.534542,172.16.115.20,32841,172.16.112.50,23,tcp,-,...,T,T,0,ShAdDFaf,26,1143,23,2388,-,6
8,f8,9.559143e+08,9.559143e+08,0.351929,172.16.115.20,32839,172.16.112.50,21,tcp,ftp,...,T,T,0,ShAdDFaf,12,562,10,663,-,6


In [44]:
src_ip = "172.16.112.50"
src_flows = df_4[df_4["src_ip"] == src_ip]
src_flows

,flow_id,start_time,end_time,duration,src_ip,sport,dst_ip,dport,proto,service,...,local_orig,local_resp,missed_bytes,history,orig_pkts,orig_ip_bytes,resp_pkts,resp_ip_bytes,tunnel_parents,ip_proto
5,f5,9.559143e+08,9.559143e+08,0.000000,172.16.112.50,33345,172.16.115.20,9325,udp,-,...,T,T,0,D,1,37,0,0,-,17
7,f7,9.559143e+08,9.559143e+08,0.166269,172.16.112.50,20,172.16.115.20,32840,tcp,ftp-data,...,T,T,0,ShAdfFa,11,444,37,45721,-,6


In [45]:
src_ips_4, dst_ips_4, src_ports_4, dst_ports_4 = analyze_phase(df_4)

Total Flows: 9

 --- IP distribution ---

Source IPs (3):
src_ip
172.16.115.20     6
172.16.112.50     2
202.77.162.213    1
Name: count, dtype: int64

Destination IPs (2):
dst_ip
172.16.112.50    6
172.16.115.20    3
Name: count, dtype: int64

 --- Port distribution ---
Source Ports (7):
sport
33368    2
33369    2
32650    1
33345    1
32841    1
20       1
32839    1
Name: count, dtype: int64

Destination Ports (6):
dport
23       2
111      2
32773    2
9325     1
32840    1
21       1
Name: count, dtype: int64


## Phase 5: Launching DDoS

In [46]:
df_5 = dfs[4]
df_5

,flow_id,start_time,end_time,duration,src_ip,sport,dst_ip,dport,proto,service,...,local_orig,local_resp,missed_bytes,history,orig_pkts,orig_ip_bytes,resp_pkts,resp_ip_bytes,tunnel_parents,ip_proto
0,f0,9.559155e+08,9.559155e+08,29.652422,172.16.115.20,32844,172.16.112.50,23,tcp,-,...,T,T,0,ShADadfF,156,6446,91,5252,-,6
1,f1,9.559155e+08,9.559155e+08,0.000000,172.16.112.50,33370,172.16.115.20,9325,udp,-,...,T,T,0,D,1,37,0,0,-,17
2,f2,9.559156e+08,9.559156e+08,0.000000,172.16.115.20,33387,172.16.112.50,7983,udp,-,...,T,T,0,D,1,50,0,0,-,17
3,f3,9.559156e+08,9.559156e+08,0.000000,172.16.112.50,3,172.16.115.20,3,icmp,-,...,T,T,0,-,1,78,0,0,-,1
4,f4,9.559155e+08,9.559160e+08,488.066625,202.77.162.213,34888,172.16.115.20,23,tcp,-,...,F,T,0,ShADad,448,23653,254,16301,-,6


In [47]:
proto_counts = df_5["proto"].value_counts()
print("Protocol distribution in Phase 5:")
print(proto_counts)

Protocol distribution in Phase 5:
proto
tcp     2
udp     2
icmp    1
Name: count, dtype: int64


In [48]:
src_ips_5, dst_ips_5, src_ports_5, dst_ports_5 = analyze_phase(df_5)

Total Flows: 5

 --- IP distribution ---

Source IPs (3):
src_ip
172.16.115.20     2
172.16.112.50     2
202.77.162.213    1
Name: count, dtype: int64

Destination IPs (2):
dst_ip
172.16.115.20    3
172.16.112.50    2
Name: count, dtype: int64

 --- Port distribution ---
Source Ports (5):
sport
32844    1
33370    1
33387    1
3        1
34888    1
Name: count, dtype: int64

Destination Ports (4):
dport
23      2
9325    1
7983    1
3       1
Name: count, dtype: int64


In [49]:
counter = 0
for ip in df_5["dst_ip"]:
    if ip.startswith("172.16."):
        counter += 1
print(f"Number of destination IPs in phase 5 that are in the homenet: {counter}")

Number of destination IPs in phase 5 that are in the homenet: 5


In [50]:
from utils import analyze_origin_destination

In [51]:
analyze_origin_destination(df_5, "Phase 5")

Phase 5 Origin Distribution:
local_orig
T    4
F    1
Name: count, dtype: int64
Phase 5 Destination Distribution:
local_resp
T    5
Name: count, dtype: int64


In [52]:
df_5_proto_counts = df_5["proto"].value_counts()
print("Protocol distribution in Phase 5:")
print(df_5_proto_counts)

Protocol distribution in Phase 5:
proto
tcp     2
udp     2
icmp    1
Name: count, dtype: int64
